In [0]:
#Load Prepared Data

from pyspark.ml.classification import (RandomForestClassifier, 
                                        DecisionTreeClassifier,
                                        LogisticRegression,
                                        GBTClassifier)
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.sql.functions import col

# Load prepared data
df = spark.read.parquet("/Volumes/workspace/default/museum/prepared/")

# Filter to top 4 kingdoms only
df = df.filter(col("kingdom").isin(["Animalia", "Protista", "Plantae", "Chromista"]))

# Train,validation,test split (70/15/15)
train, val, test = df.randomSplit([0.7, 0.15, 0.15], seed=42)

print(f"Training rows: {train.count():,}")
print(f"Validation rows: {val.count():,}")
print(f"Test rows: {test.count():,}")

Training rows: 800,348
Validation rows: 171,338
Test rows: 171,667


In [0]:
#Random Forest
rf = RandomForestClassifier(maxBins=6000,
    featuresCol="features",
    labelCol="label",
    numTrees=50,
    maxDepth=10,
    seed=42
)

rf_model = rf.fit(train)
rf_preds = rf_model.transform(test)

evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction")

rf_accuracy = evaluator.evaluate(rf_preds, {evaluator.metricName: "accuracy"})
rf_f1 = evaluator.evaluate(rf_preds, {evaluator.metricName: "f1"})

print(f"Random Forest Accuracy: {rf_accuracy:.4f}")
print(f"Random Forest F1 Score: {rf_f1:.4f}")


Random Forest Accuracy: 0.9998
Random Forest F1 Score: 0.9998


In [0]:
#Decision Tree
dt = DecisionTreeClassifier(
    featuresCol="features",
    labelCol="label",
    maxDepth=10,
    maxBins=6000,
    seed=42
)

dt_model = dt.fit(train)
dt_preds = dt_model.transform(test)

dt_accuracy = evaluator.evaluate(dt_preds, {evaluator.metricName: "accuracy"})
dt_f1 = evaluator.evaluate(dt_preds, {evaluator.metricName: "f1"})

print(f"Decision Tree Accuracy: {dt_accuracy:.4f}")
print(f"Decision Tree F1 Score: {dt_f1:.4f}")

Decision Tree Accuracy: 0.9987
Decision Tree F1 Score: 0.9987


In [0]:
#Logistic Regression

lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=10,
    regParam=0.01
)

lr_model = lr.fit(train)
lr_preds = lr_model.transform(test)

lr_accuracy = evaluator.evaluate(lr_preds, {evaluator.metricName: "accuracy"})
lr_f1 = evaluator.evaluate(lr_preds, {evaluator.metricName: "f1"})

print(f"Logistic Regression Accuracy: {lr_accuracy:.4f}")
print(f"Logistic Regression F1 Score: {lr_f1:.4f}")

Logistic Regression Accuracy: 0.9601
Logistic Regression F1 Score: 0.9532


In [0]:
 #GBT (binary only - Animalia vs non-Animalia)
from pyspark.sql.functions import when

train_binary = train.withColumn("label", when(col("label") == 0.0, 1.0).otherwise(0.0))
test_binary = test.withColumn("label", when(col("label") == 0.0, 1.0).otherwise(0.0))

gbt = GBTClassifier(
    featuresCol="features",
    labelCol="label",
    maxIter=20,
    maxDepth=5,
    maxBins=6000,
    seed=42
)

gbt_model = gbt.fit(train_binary)
gbt_preds = gbt_model.transform(test_binary)

binary_evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction")
gbt_accuracy = binary_evaluator.evaluate(gbt_preds, {binary_evaluator.metricName: "accuracy"})
gbt_f1 = binary_evaluator.evaluate(gbt_preds, {binary_evaluator.metricName: "f1"})

print(f"GBT Accuracy (Animalia vs rest): {gbt_accuracy:.4f}")
print(f"GBT F1 Score (Animalia vs rest): {gbt_f1:.4f}")

GBT Accuracy (Animalia vs rest): 0.9996
GBT F1 Score (Animalia vs rest): 0.9996


In [0]:
import os
os.environ['SPARKML_TEMP_DFS_PATH'] = '/Volumes/workspace/default/museum/tmp/'

# Hyperparameter tuning on Random Forest
rf_tuning = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    maxBins=6000,
    seed=42
)

paramGrid = ParamGridBuilder() \
    .addGrid(rf_tuning.numTrees, [20, 50]) \
    .addGrid(rf_tuning.maxDepth, [5, 10]) \
    .build()

crossval = CrossValidator(
    estimator=rf_tuning,
    estimatorParamMaps=paramGrid,
    evaluator=MulticlassClassificationEvaluator(labelCol="label", metricName="accuracy"),
    numFolds=3,
    parallelism=2
)

cv_model = crossval.fit(train)
cv_preds = cv_model.transform(test)

cv_accuracy = evaluator.evaluate(cv_preds, {evaluator.metricName: "accuracy"})
cv_f1 = evaluator.evaluate(cv_preds, {evaluator.metricName: "f1"})

print(f"Tuned Random Forest Accuracy: {cv_accuracy:.4f}")
print(f"Tuned Random Forest F1: {cv_f1:.4f}")
print(f"Best num trees: {cv_model.bestModel.getNumTrees}")
print(f"Best max depth: {cv_model.bestModel.getMaxDepth()}")


Tuned Random Forest Accuracy: 0.9998
Tuned Random Forest F1: 0.9998
Best num trees: 50
Best max depth: 10


In [0]:
#Save Models and Results Summary


model_path = "/Volumes/workspace/default/museum/models/"



rf_model.write().overwrite().save(model_path + "random_forest")
dt_model.write().overwrite().save(model_path + "decision_tree")
lr_model.write().overwrite().save(model_path + "logistic_regression")
gbt_model.write().overwrite().save(model_path + "gbt")
cv_model.bestModel.write().overwrite().save(model_path + "random_forest_tuned")


# Results summary table
results = spark.createDataFrame([
    ("Random Forest",       rf_accuracy,  rf_f1),
    ("Decision Tree",       dt_accuracy,  dt_f1),
    ("Logistic Regression", lr_accuracy,  lr_f1),
    ("GBT (binary)",        gbt_accuracy, gbt_f1),
    ("Tuned Random Forest", cv_accuracy,  cv_f1),
], ["Model", "Accuracy", "F1 Score"])

display(results.orderBy("Accuracy", ascending=False))
print("All models saved successfully")

All models saved successfully


Model,Accuracy,F1 Score
Random Forest,0.9998427187520024,0.9998411663070507
Tuned Random Forest,0.9998427187520024,0.9998411663070507
GBT (binary),0.9995747581072658,0.9995749812146013
Decision Tree,0.9986893229333536,0.99870400731676
Logistic Regression,0.9600796891656521,0.9531552173277824


All models saved successfully


In [0]:
#TRAINING TIME & MEMORY USAGE 

import time
import psutil
import os

# Measure training times
results = []

# Random Forest
start = time.time()
rf_temp = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=50, maxDepth=10, maxBins=6000, seed=42).fit(train)
rf_time = round(time.time() - start, 2)

# Decision Tree
start = time.time()
dt_temp = DecisionTreeClassifier(featuresCol="features", labelCol="label", maxDepth=10, maxBins=6000, seed=42).fit(train)
dt_time = round(time.time() - start, 2)

# Logistic Regression
start = time.time()
lr_temp = LogisticRegression(featuresCol="features", labelCol="label", maxIter=10, regParam=0.01).fit(train)
lr_time = round(time.time() - start, 2)

# GBT
start = time.time()
gbt_temp = GBTClassifier(featuresCol="features", labelCol="label", maxIter=20, maxDepth=5, maxBins=6000, seed=42).fit(train_binary)
gbt_time = round(time.time() - start, 2)

print(f"Random Forest training time: {rf_time}s")
print(f"Decision Tree training time: {dt_time}s")
print(f"Logistic Regression training time: {lr_time}s")
print(f"GBT training time: {gbt_time}s")

# Export
import pandas as pd
timing_df = pd.DataFrame({
    "Model": ["Random Forest", "Decision Tree", "Logistic Regression", "GBT"],
    "Training_Time_Seconds": [rf_time, dt_time, lr_time, gbt_time],
    "Accuracy": [0.9998, 0.9987, 0.9601, 0.9996]
})
timing_df.to_csv("/Volumes/workspace/default/museum/tableau/training_times.csv", index=False)


Random Forest training time: 124.12s
Decision Tree training time: 14.58s
Logistic Regression training time: 10.62s
GBT training time: 61.83s
Training times exported
